In [13]:
# PyAST_test.py
from keymaerax_parser import KeYmaeraXParser, convert_keymaerax_ast_to_intermediate_pyast

# --- KeYmaera X Input ---
kx_input = """
Theorem "IJRR17/Theorem 1: Static safety"

Definitions
  Real ep;      /* time limit for control decisions */
  Real b;       /* minimum braking capability of the robot */
  Real A;       /* maximum acceleration -b <= a <= A */
  Real W;       /* maximum steering */

  Real stopDist(Real v) = v^2 / (2*b);
  Real accelComp(Real v) = ((A/b + 1) * (A/2 * ep^2 + ep*v));
  Real admissibleSeparation(Real v) = stopDist(v) + accelComp(v);

  Bool isWellformedDir(Real dx, Real dy) <-> dx^2 + dy^2 = 1;

  Bool bounds() <-> (
      A >= 0
    & b > 0
    & ep > 0
  );
  
  /* control */
  HP controlFunc ::= {
      /* brake on current curve or remain stopped */
      { a := -b; }
      ++
      { ?v = 0; a := 0; w := 0; }
      ++
      /* or choose a new safe curve */
      { a := A;
        w := *; ?-W<=w & w<=W;
        r := *;
        xo := *; yo := *;
        ?r!=0 & r*w = v;
        ? abs(x-xo) > admissibleSeparation(v)
        | abs(y-yo) > admissibleSeparation(v);
      }
    };;

    HP stepFunction ::= { 
        x' = v * dx, y' = v * dy, v' = a,
        dx' = -w * dy, dy' = w * dx, w' = a/r,
        t' = 1 & t <= ep & v >= 0
      };
End.

ProgramVariables
  Real x;    /* robot position: x */
  Real y;    /* robot position: y */
  Real v;    /* robot translational velocity */
  Real a;    /* robot translational acceleration */
  Real dx;   /* robot orientation: x */
  Real dy;   /* robot orientation: y */
  Real w;    /* robot rotational velocity */
  Real r;    /* robot curve radius */
  Real xo;   /* position of closest obstacle on curve */
  Real yo;
  Real t;    /* time */
End.

Problem
assumptions(x, y, v, dx, dy, xo, yo) ->
  [
    {
      {
        controlFunc;
        t := 0;
      }

      /* dynamics */
      {
      stepFunction;@invariant(t>=0, isWellformedDir(dx, dy))
      }
    }*@invariant(loopinv(x, y, v, dx, dy, xo, yo))
  ](x - xo)^2 + (y - yo)^2 > 0
End.
"""

# --- Main Execution ---
if __name__ == "__main__":
    print("--- Parsing KeYmaera X Input into internal AST ---")
    parser = KeYmaeraXParser(kx_input)
    kx_ast = parser.parse()
    # Print the internal KeYmaera X AST for debugging/verification
    print(kx_ast)

    print("\n--- Converting KeYmaera X AST to Intermediate Python AST ---")
    py_ast_config = convert_keymaerax_ast_to_intermediate_pyast(kx_ast)
    # Print the resulting PyAST configuration
    print(py_ast_config)

--- Parsing KeYmaera X Input into internal AST ---
[KeYmaeraXSpecification]
  [TheoremName]: IJRR17/Theorem 1: Static safety
  [DefinitionsBlock]
    [HP]: controlFunc
      Body (raw): { { a := -b; } ++ { ?v = 0; a := 0; w := 0; } ++ { a := A; w := *; ?-W<=w & w<=W...
      [ChoiceOperator (++)]
        [Choice 1]
          [Statement]: a := -b;
        [Choice 2]
          [Test]: ?v = 0
          [Statement]: a := 0
          [Statement]: w := 0
        [Choice 3]
          [Statement]: a := A
          [Statement]: w := *
          [Test]: ?-W<=w & w<=W
          [Statement]: r := *
          [Statement]: xo := *
          [Statement]: yo := *
          [Test]: ?r!=0 & r*w = v
          [Test]: ? abs(x-xo) > admissibleSeparation(v)
        | abs(y-yo) > admissibleSeparation(v)
    [HP]: stepFunction
      Body (raw): { x' = v * dx, y' = v * dy, v' = a, dx' = -w * dy, dy' = w * dx, w' = a/r, t' = ...
    [DEF]: Real ep
    [DEF]: Real b
    [DEF]: Real A
    [DEF]: Real W
    [DEF]: